# Baseline Modeling 

This notebook trains and evaluates **baseline predictive models** using the feature table created in **Notebook 02 (Data Preparation & Feature Engineering)**.

## Inputs and outputs

**Input**
- `df_feat` (Parquet): cleaned + engineered dataset including:
  - Predictors (primarily Year 1 features)
  - Targets/labels (Year 2 outcomes), e.g.:
    - Regression: `LOG_TOTEXPY2`
    - Classification: `HIGHCOST_Y2`, `ANY_ED_Y2`, `ANY_IP_Y2`

**Outputs**
- Baseline model performance tables (validation + test metrics)
- Saved figures/tables under `results/`



## 0) Setup and imports

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Make project importable (so we can `import src.*`) ---
PROJECT_ROOT = Path.cwd().resolve().parents[0]  
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/wenxi/Desktop/TFM_25


## 1) Load the feature dataset (`df_feat`)

In [1]:
import pandas as pd

df_feat = pd.read_parquet("../data/df_feat.parquet")


## 2) Define prediction tasks (targets)

**Goal:** Make the modeling tasks explicit and consistent.

Typical tasks:
- **Regression**
  - `y_reg = LOG_TOTEXPY2` (log1p-transformed Year 2 total expenditure)
- **Classification**
  - `y_highcost = HIGHCOST_Y2` (top 10% cost indicator)
  - `y_ed = ANY_ED_Y2` (any ED visit)
  - `y_ip = ANY_IP_Y2` (any inpatient stay)


In [7]:
TARGET_COLS = ["TOTEXPY2", "LOG_TOTEXPY2", "HIGHCOST_Y2", "ANY_ED_Y2", "ANY_IP_Y2"]

## 3) Define the feature set (X) and prevent leakage

**Goal:** Build a predictor matrix that does not leak Year 2 information.

- Exclude:
  - IDs (e.g., `DUPERSID`, `DUID`, `PID`)
  - survey design variables / weights unless explicitly modeling with them
  - all target columns and any Year 2 utilization/cost variables that directly encode outcomes
- Keep:
  - Year 1 baseline cost/utilization features (predictors)
  - demographics, SES, insurance, health/chronic, employment features (mostly Year 1)

In [6]:
import numpy as np
import pandas as pd

# 1) define columns you NEVER want as model predictors
ID_COLS = ["DUPERSID", "DUID", "PID", "PANEL", "VARSTR", "VARPSU"]
WEIGHT_COLS = ["LONGWT", "LSAQWT"]

# targets 
TARGET_COLS = ["TOTEXPY2", "LOG_TOTEXPY2", "HIGHCOST_Y2", "ANY_ED_Y2", "ANY_IP_Y2"]

EXCLUDE = set(ID_COLS + WEIGHT_COLS + TARGET_COLS)

# 2) candidate feature columns = everything else
feature_candidates = [c for c in df_feat.columns if c not in EXCLUDE]

# 3) categorical = object/category
cat_cols = df_feat[feature_candidates].select_dtypes(include=["object", "category"]).columns.tolist()

# 4) numeric = number types (int/float/bool)
num_cols = df_feat[feature_candidates].select_dtypes(include=[np.number]).columns.tolist()

print("n feature candidates:", len(feature_candidates))
print("n cat:", len(cat_cols))
print("n num:", len(num_cols))

cat_cols[:20], num_cols[:20]


n feature candidates: 128
n cat: 7
n num: 121


(['AGE_GROUP',
  'RACE_ETH',
  'REGIONY1_CAT',
  'EDU_GROUP',
  'POVCATY1_CAT',
  'FAMSIZE_Y1_GRP',
  'INS_TYPE_Y1'],
 ['YEARIND',
  'ALL5RDS',
  'DIED',
  'INST',
  'MILITARY',
  'ENTRSRVY',
  'LEFTUS',
  'OTHER',
  'AGEY1X',
  'AGEY2X',
  'AGELSTY1',
  'AGELSTY2',
  'SEX',
  'RACETHX',
  'HISPANX',
  'EDUCYR',
  'REGIONY1',
  'REGIONY2',
  'FAMINCY1',
  'FAMINCY2'])

In [8]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]

In [9]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

   

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]

## 4) Implementation note (splitting + preprocessing)

The **train/validation/test split** and the **preprocessing pipeline** are implemented as reusable functions in `src/models.py`, rather than being written inline in this notebook.

- **Data splitting** is handled inside the baseline runners (e.g., `run_regression_baseline` and `run_classification_baseline`), using a fixed `random_state` for reproducibility. For classification tasks, stratification is applied when appropriate.
- **Preprocessing** is built via a `ColumnTransformer` inside `src/models.py`:
  - Numeric features: median imputation (and optional scaling)
  - Categorical features: most-frequent imputation + one-hot encoding (`handle_unknown="ignore"`)
  - The transformer is **fit on the training set only**, then applied to validation/test sets to avoid leakage.

In this notebook, we therefore focus on defining:
- the target column (`target_col`)
- the predictor column lists (`num_cols`, `cat_cols`)
and then call the baseline functions from `src/models.py` to train and evaluate models consistently across tasks.


In [5]:
from src.models import  make_preprocess, split_train_val_test, run_regression_baseline, best_threshold_by_f1, run_classification_baseline

## 5) Baseline regression model (Year 2 cost)

**Goal:** Establish a simple, interpretable baseline for cost prediction.

Baseline:
- **ElasticNet** on `LOG_TOTEXPY2` (handles collinearity and high-dimensional one-hot features)

Report on validation and test:
- R²
- RMSE (on log scale)
- MAE

Hyperparameter tuning on training set (CV over alpha and l1_ratio), then select based on validation performance.


In [16]:
# Regression baseline with a few key predictors (age, baseline cost, baseline utilisation, insurance type)
reg_res = run_regression_baseline(
    df_feat,
    target_col="LOG_TOTEXPY2",
    num_cols=["AGE","LOG_TOTEXPY1","ANY_ED_Y1","ANY_IP_Y1"],
    cat_cols=["INS_TYPE_Y1"],
    alpha=0.01,
    l1_ratio=0.5,
)

print("REG VALID:", reg_res.valid_metrics)
print("REG TEST:",  reg_res.test_metrics)  

REG VALID: {'MAE_log': 1.7018088538418008, 'RMSE_log': 2.348043515579074, 'R2': 0.48321311002065914}
REG TEST: {'MAE_log': 1.5856456859650883, 'RMSE_log': 2.2385298204061734, 'R2': 0.49023398173473554}


In [10]:
# Regression (LOG_TOTEXPY2) with all candidate features 
reg_res = run_regression_baseline(
    df_feat,
    target_col="LOG_TOTEXPY2",
    num_cols=num_cols,
    cat_cols=cat_cols,
    alpha=0.01,
    l1_ratio=0.5,
    
)

print("REG VALID:", reg_res.valid_metrics)



REG VALID: {'MAE_log': 1.6572547698657458, 'RMSE_log': 2.2768887072679838, 'R2': 0.5140598186235315}


### Results (validation set)
For the baseline regression on `LOG_TOTEXPY2`, we obtain:

- **R² ≈ 0.51**  
  → The model explains about **51%** of the variation in Year-2 log expenditures on the validation set.

- **MAE_log ≈ 1.66**, **RMSE_log ≈ 2.28** (log scale)  
  → Errors are reported on the **log(1 + cost)** scale, which is standard for highly right-skewed cost data.
  Lower is better; RMSE penalizes large errors more than MAE.

Overall, this provides a reasonable baseline reference point for later, more flexible models.

### Hyperparameter tuning (use validation)

In [11]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.metrics import  root_mean_squared_error,r2_score

# 1) build X, y
X = df_feat[num_cols + cat_cols]
y = df_feat["LOG_TOTEXPY2"]

# 2) split
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=42, stratify=False
)

# 3) build preprocess
preprocess = make_preprocess(num_cols, cat_cols, scale_numeric=True)

# 4) pipeline
reg_model = Pipeline([
    ("preprocess", preprocess),
    ("model", ElasticNet(max_iter=20000, random_state=42)),
])

# 5) grid search on TRAIN only
param_grid = {
    "model__alpha": np.logspace(-4, -1, 7),
    "model__l1_ratio": [0.1, 0.5, 0.9],
}

gs = GridSearchCV(
    reg_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
gs.fit(X_train, y_train)

best_reg = gs.best_estimator_
print("Best params:", gs.best_params_)

# 6) evaluate on VAL
val_pred = best_reg.predict(X_val)
val_rmse = root_mean_squared_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)

print("VAL RMSE_log:", val_rmse)
print("VAL R2:", val_r2)


Best params: {'model__alpha': np.float64(0.001), 'model__l1_ratio': 0.9}
VAL RMSE_log: 2.2786428126197578
VAL R2: 0.513310797725858


Tuning ElasticNet (`alpha`, `l1_ratio`) did not materially improve validation RMSE/R², so we keep the original baseline configuration as the reference model.


### Validation vs. test performance (regression)

In [12]:
# Regression (LOG_TOTEXPY2) with all candidate features 
reg_res = run_regression_baseline(
    df_feat,
    target_col="LOG_TOTEXPY2",
    num_cols=num_cols,
    cat_cols=cat_cols,
    alpha=0.01,
    l1_ratio=0.5,
)

print("REG VALID:", reg_res.valid_metrics)
print("REG TEST:",  reg_res.test_metrics)   




REG VALID: {'MAE_log': 1.6572547698657458, 'RMSE_log': 2.2768887072679838, 'R2': 0.5140598186235315}
REG TEST: {'MAE_log': 1.5553245943844185, 'RMSE_log': 2.1653297844655826, 'R2': 0.5230276404428619}


The ElasticNet baseline with a small feature set (age, prior cost, prior utilization, insurance type) achieves moderate performance (test RMSE_log ≈ 2.24, R² ≈ 0.49). Expanding to the full candidate feature set improves fit consistently on both validation and test, with lower error (test RMSE_log ≈ 2.17) and higher explained variance (test R² ≈ 0.52). Overall, the richer feature set yields a modest but clear gain in predictive accuracy.

The baseline also shows **similar performance on the validation and test sets**, suggesting good generalization .

- **Validation:** RMSE_log ≈ 2.28, MAE_log ≈ 1.66, R² ≈ 0.51  
- **Test:** RMSE_log ≈ 2.17, MAE_log ≈ 1.56, R² ≈ 0.52  

 Overall, the close agreement between validation and test metrics indicates that the baseline model is stable and provides a reliable reference point for later, more complex approaches.


## 6) Baseline classification models (high-cost, ED, inpatient)

**Goal:** Establish baselines for binary event prediction with imbalanced labels.

Baseline:
- **Logistic Regression**
  - Use `class_weight="balanced"` (or compare with unweighted)

Report on validation and test:
- ROC AUC
- PR-AUC (more informative under class imbalance)



### Validation vs. test performance (classification)

In [17]:
#classification baselines (HIGHCOST_Y2, ANY_ED_Y2, ANY_IP_Y2) with a few key predictors (age, baseline cost, baseline utilisation, insurance type)

hc_res = run_classification_baseline(df_feat, target_col="HIGHCOST_Y2", num_cols=["AGE","LOG_TOTEXPY1","ANY_ED_Y1","ANY_IP_Y1"], cat_cols=["INS_TYPE_Y1"])
ed_res = run_classification_baseline(df_feat, target_col="ANY_ED_Y2",   num_cols=["AGE","LOG_TOTEXPY1","ANY_ED_Y1","ANY_IP_Y1"], cat_cols=["INS_TYPE_Y1"])
ip_res = run_classification_baseline(df_feat, target_col="ANY_IP_Y2",   num_cols=["AGE","LOG_TOTEXPY1","ANY_ED_Y1","ANY_IP_Y1"], cat_cols=["INS_TYPE_Y1"])

print("HC VALID:", hc_res.valid_metrics)
print("HC TEST:",  hc_res.test_metrics)     

print("ED VALID:", ed_res.valid_metrics)
print("ED TEST:",  ed_res.test_metrics)     

print("IP VALID:", ip_res.valid_metrics)
print("IP TEST:",  ip_res.test_metrics)

HC VALID: {'AUC': 0.8208904512960833, 'PR_AUC': 0.3882723581704644, 'best_t': np.float64(0.7), 'best_F1': 0.4343163538873995}
HC TEST: {'AUC': 0.8626214167258943, 'PR_AUC': 0.46641775653130746, 'F1_at_best_t': 0.5301837270341208}
ED VALID: {'AUC': 0.725406599290543, 'PR_AUC': 0.3496862633907676, 'best_t': np.float64(0.6), 'best_F1': 0.40150093808630394}
ED TEST: {'AUC': 0.697995448765143, 'PR_AUC': 0.3137795053986525, 'F1_at_best_t': 0.3187250996015936}
IP VALID: {'AUC': 0.7881418522150787, 'PR_AUC': 0.29750233473373733, 'best_t': np.float64(0.7), 'best_F1': 0.35374149659863946}
IP TEST: {'AUC': 0.7520384498016479, 'PR_AUC': 0.2034082671517488, 'F1_at_best_t': 0.26865671641791045}


In [13]:
# Classification baselines (HIGHCOST_Y2, ANY_ED_Y2, ANY_IP_Y2) with all candidate features
hc_res = run_classification_baseline(df_feat, target_col="HIGHCOST_Y2", num_cols=num_cols, cat_cols=cat_cols)
ed_res = run_classification_baseline(df_feat, target_col="ANY_ED_Y2",   num_cols=num_cols, cat_cols=cat_cols)
ip_res = run_classification_baseline(df_feat, target_col="ANY_IP_Y2",   num_cols=num_cols, cat_cols=cat_cols)

print("HC VALID:", hc_res.valid_metrics)
print("HC TEST:",  hc_res.test_metrics)     

print("ED VALID:", ed_res.valid_metrics)
print("ED TEST:",  ed_res.test_metrics)     

print("IP VALID:", ip_res.valid_metrics)
print("IP TEST:",  ip_res.test_metrics)     

HC VALID: {'AUC': 0.8166049052740303, 'PR_AUC': 0.343884020637643, 'best_t': np.float64(0.6), 'best_F1': 0.41282565130260523}
HC TEST: {'AUC': 0.8496756146009878, 'PR_AUC': 0.4238439255285054, 'F1_at_best_t': 0.44952380952380955}
ED VALID: {'AUC': 0.7385583294290878, 'PR_AUC': 0.3670506610771668, 'best_t': np.float64(0.65), 'best_F1': 0.4149377593360996}
ED TEST: {'AUC': 0.7066193695201125, 'PR_AUC': 0.33085901207493673, 'F1_at_best_t': 0.37333333333333335}
IP VALID: {'AUC': 0.7902182993716174, 'PR_AUC': 0.2803399591324149, 'best_t': np.float64(0.7999999999999999), 'best_F1': 0.36607142857142855}
IP TEST: {'AUC': 0.7506499847421422, 'PR_AUC': 0.21956377090688456, 'F1_at_best_t': 0.27722772277227725}


Using a small set of key predictors already yields strong baseline performance. Adding all candidate features slightly improves ED and IP prediction but provides little benefit for HIGHCOST (and even degrades it). Validation and test metrics are close, suggesting stable generalization with no major overfitting.



#### High-cost (HIGHCOST_Y2)
- **Validation:** AUC ≈ 0.82, PR-AUC ≈ 0.34, best F1 ≈ 0.41  
- **Test:** AUC ≈ 0.85, PR-AUC ≈ 0.42, F1@best-threshold ≈ 0.45  
**Interpretation:** This is the strongest of the three tasks. The model shows good discrimination and improved precision–recall performance on the test set.

#### Any ED visit (ANY_ED_Y2)
- **Validation:** AUC ≈ 0.74, PR-AUC ≈ 0.37, best F1 ≈ 0.41  
- **Test:** AUC ≈ 0.71, PR-AUC ≈ 0.33, F1@best-threshold ≈ 0.37  
**Interpretation:** Moderate predictive ability. Test performance is slightly lower than validation, which is consistent with normal split-to-split variation.

#### Any inpatient stay (ANY_IP_Y2)
- **Validation:** AUC ≈ 0.79, PR-AUC ≈ 0.28, best F1 ≈ 0.37  
- **Test:** AUC ≈ 0.75, PR-AUC ≈ 0.22, F1@best-threshold ≈ 0.28  
**Interpretation:** This is the most challenging task, likely due to stronger class imbalance and lower event prevalence. Discrimination is acceptable (AUC), but PR-AUC and F1 are lower, indicating limited precision at useful recall levels.

**Overall conclusion:** The baseline Logistic Regression performs best on **high-cost prediction**, moderately on **ED visits**, and worst on **inpatient events**. These results provide a clear reference point for later models and justify using PR-AUC and imbalance-aware strategies (e.g., class weights, threshold calibration, and top-k evaluation).


## 7） Save baseline results

In [14]:
import pandas as pd

rows = [
    {"Task": "Regression (LOG_TOTEXPY2)", "Split": "VALID", **reg_res.valid_metrics},
    {"Task": "Regression (LOG_TOTEXPY2)", "Split": "TEST",  **reg_res.test_metrics},

    {"Task": "High-cost (HIGHCOST_Y2)", "Split": "VALID", **hc_res.valid_metrics},
    {"Task": "High-cost (HIGHCOST_Y2)", "Split": "TEST",  **hc_res.test_metrics},
 
    {"Task": "Any ED (ANY_ED_Y2)", "Split": "VALID", **ed_res.valid_metrics},
    {"Task": "Any ED (ANY_ED_Y2)", "Split": "TEST",  **ed_res.test_metrics},

    {"Task": "Any IP (ANY_IP_Y2)", "Split": "VALID", **ip_res.valid_metrics},
    {"Task": "Any IP (ANY_IP_Y2)", "Split": "TEST",  **ip_res.test_metrics},
]

results_table = pd.DataFrame(rows)
results_table


,Task,Split,MAE_log,RMSE_log,R2,AUC,PR_AUC,best_t,best_F1,F1_at_best_t
0,Regression (LOG_TOTEXPY2),VALID,1.657255,2.276889,0.514060,NaN,NaN,NaN,NaN,NaN
1,Regression (LOG_TOTEXPY2),TEST,1.555325,2.165330,0.523028,NaN,NaN,NaN,NaN,NaN
2,High-cost (HIGHCOST_Y2),VALID,NaN,NaN,NaN,0.816605,0.343884,0.60,0.412826,NaN
3,High-cost (HIGHCOST_Y2),TEST,NaN,NaN,NaN,0.849676,0.423844,NaN,NaN,0.449524
4,Any ED (ANY_ED_Y2),VALID,NaN,NaN,NaN,0.738558,0.367051,0.65,0.414938,NaN
5,Any ED (ANY_ED_Y2),TEST,NaN,NaN,NaN,0.706619,0.330859,NaN,NaN,0.373333
6,Any IP (ANY_IP_Y2),VALID,NaN,NaN,NaN,0.790218,0.280340,0.80,0.366071,NaN
7,Any IP (ANY_IP_Y2),TEST,NaN,NaN,NaN,0.750650,0.219564,NaN,NaN,0.277228


In [15]:
from src.config import TABLES_DIR

# make sure the tables directory exists
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# choose an output filename
out_csv = TABLES_DIR / "baseline_results.csv"
results_table.to_csv(out_csv, index=False)

print("Saved table to:", out_csv)

Saved table to: /Users/wenxi/Desktop/TFM_25/results/tables/baseline_results.csv
